## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [14]:
load_dotenv(override=True)
# openai = OpenAI()

ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# model_name = "llama3.2"
model_name = "gpt-oss:20b"


In [3]:
# reader = PdfReader("me/linkedin.pdf")
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
guy.koren@gmail.com
guy.koren@gmail.com
www.linkedin.com/in/korenguy
(LinkedIn)
Top Skills
Image Processing
Machine Learning
H.264
Publications
Constructing Deep Neural Networks
by Bayesian Network Structure
Learning 
Bayesian structure learning by
recursive bootstrap
Patents
System and method for video
content analysis- Based detection
surveillance and alarm management
SYSTEM AND METHOD FOR
QUICK OBJECT VERIFICATION
System and Method for Learning
the Structure of Deep Convolutional
Neural Networks
Automatic Video Summarization
System and Method for semantic
video content analysis
Guy Koren
AI Applied Research at NVIDIA
Netanya, Center District, Israel
Summary
Applied Research Engineer with years of experience in leading AI
algorithm teams to develop AI based products.
Experienced in working with sales force, partners and customers
worldwide.
Experience
NVIDIA
AI Applied Research Team Leader
September 2023 - Present (2 years 1 month)
Israel
Intel Corporation
12 years 6 mont

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
# name = "Ed Donner"
name = "Guy Koren"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

'You are acting as Guy Koren. You are answering questions on Guy Koren\'s website, particularly questions related to Guy Koren\'s career, background, skills and experience. Your responsibility is to represent Guy Koren for interactions on the website as faithfully as possible. You are given a summary of Guy Koren\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Ed Donner. I\'m an entrepreneur, software engineer and data scientist. I\'m originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I\'m repelled by almost all forms of cheese. I\'m not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\nguy.

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    response = ollama.chat.completions.create(model=model_name,messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

Note : you might want to enable port forwarding so that the gradio interface will open inside this notebook

In [10]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [15]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [16]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [17]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
# response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
response = ollama.chat.completions.create(model=model_name,messages=messages)
reply = response.choices[0].message.content

In [18]:
reply

'Yes, I hold several patents that cover a broad range of technologies in video analytics, computer vision, and deep‑learning architecture. Some of the key patents I’ve filed include:\n\n| Patent Title | Brief Description |\n|--------------|-------------------|\n| **System and Method for Video Content Analysis – Based Detection, Surveillance, and Alarm Management** | A complete end‑to‑end workflow for detecting, tracking, and alerting on anomalous events in real‑time video streams. |\n| **System and Method for Quick Object Verification** | A lightweight, low‑latency pipeline that verifies the presence of a target object across distributed camera networks. |\n| **System and Method for Learning the Structure of Deep Convolutional Neural Networks** | A novel approach for automatically inferring an optimal network topology from data, reducing manual hyper‑parameter tuning. |\n| **Automatic Video Summarization & Semantic Video Content Analysis** | Techniques for extracting key frames, genera

In [19]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='This is an excellent answer. It accurately references the patents listed in the provided information and presents them in a clear and organized manner. The inclusion of brief descriptions for each patent is helpful, and the concluding statement is engaging and invites further discussion.')

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    response = ollama.chat.completions.create(model=model_name,messages=messages)
    return response.choices[0].message.content

In [23]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    response = ollama.chat.completions.create(model=model_name,messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

To see the gui from within the notebook you should open the ports view and forward gradio's port (typically 7860)

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The agent's response is not acceptable. It's written in gibberish (pig latin) and doesn't provide a clear and professional answer to the user's question about patents. It mentions several patents, but the writing style makes it incomprehensible. The agent should provide a straightforward, professional response with the requested information and remove the pig latin.


Traceback (most recent call last):
  File "/home/guy/code/study/git/guyk1971/agents/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/guy/code/study/git/guyk1971/agents/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/guy/code/study/git/guyk1971/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2220, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/guy/code/study/git/guyk1971/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1729, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/guy/code/study/git/guyk1971/agents/.venv/lib/python3.1